In [ ]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go

sys.path.insert(0, os.path.abspath('..'))

from model import ResAtt3DUNet
from hyperparameters import IN_CH, OUT_CH, NUM_FILTERS, NUM_HEADS

device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_str)

In [ ]:
# hyperparameters
in_ch = IN_CH
out_ch = OUT_CH
num_filters = NUM_FILTERS
num_heads = NUM_HEADS

# model setup
model = ResAtt3DUNet(
    in_channels=in_ch, out_channels=out_ch, num_filters=num_filters, num_heads=num_heads, dropout=False
)

model.to(device)

model.load_state_dict(torch.load('/home/omkos333/projects/brainseg/weights/final_weights.pt', map_location=device))

# Load a test image and its mask
image = np.load('/home/omkos333/projects/brainseg/data/processed/test/images/image_0.npy') # (3, 64, 64, 64)
mask = np.load('/home/omkos333/projects/brainseg/data/processed/test/masks/mask_0.npy') # (4, 64, 64, 64)

# convert to tensors
image_tensor = torch.from_numpy(image).unsqueeze(0).float().to(device) # (1, 3, 64, 64, 64)
mask_tensor = torch.from_numpy(mask).float() # (4, 64, 64, 64)

# get the ground truth mask
true_mask = torch.argmax(mask_tensor, dim=0).numpy() # (64, 64, 64)

# get the predicted mask
model.eval()

with torch.no_grad():
    logits = model(image_tensor) # outputs a predicted mask, (1, 4, 64, 64, 64)
    probs = torch.softmax(logits, dim=1)
    pred_mask = torch.argmax(probs, dim=1).squeeze(0).cpu().numpy() # (64, 64, 64)

In [ ]:
# pick a slice
slice_idx = 9
true_slice = true_mask[slice_idx]
pred_slice = pred_mask[slice_idx]

# plots
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.imshow(true_slice, cmap='viridis')
plt.title('ground truth')

plt.subplot(1, 2, 2)
plt.imshow(pred_slice, cmap='viridis')
plt.title('prediction')

plt.tight_layout()
plt.show()

In [ ]:
# 3D plot of the ground truth mask and predicted mask
def plot_mask_3D(mask, title):
    d, h, w = mask.shape[0], mask.shape[1], mask.shape[2]

    fig = go.Figure(
                go.Volume(
                    x=np.arange(w).repeat(d * h), y=np.tile(np.arange(h).repeat(d), w), z=np.tile(np.arange(d), h * w),
                    value=mask.flatten(), isomin=1, isomax=np.max(mask),
                    opacity=0.4, surface_count=10, colorscale='Viridis'
                )
    )

    fig.update_layout(
        title=title, scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'), margin=dict(t=40, l=0, b=0, r=0)
    )

    fig.show()

plot_mask_3D(true_mask, 'ground truth')
plot_mask_3D(pred_mask, 'prediction')

In [ ]:
# 3D plot of the ground truth mask and predicted mask with the brain overlayed
def plot_image_with_mask_3D(image, mask, title, channel):
    fig = go.Figure()
    
    volume = image[channel] # (64, 64, 64)
    d, h, w = volume.shape[0], volume.shape[1], volume.shape[2]
    
    fig.add_trace(
            go.Volume(
                x=np.arange(d).repeat(d * h), y=np.tile(np.arange(h).repeat(d), w), z=np.tile(np.arange(d), h * w),
                value=volume.flatten(), isomin=np.percentile(volume, 10), isomax=np.percentile(volume, 90),
                opacity=0.1, surface_count=5, colorscale='Greys'
            )
    )
    
    fig.add_trace(
            go.Volume(
                x=np.arange(w).repeat(d * mask.shape[1]), y=np.tile(np.arange(h).repeat(d), w), z=np.tile(np.arange(d), h * w),
                value=mask.flatten(), isomin=1, isomax=np.max(mask),
                opacity=0.8, surface_count=15, colorscale='Reds',
            )
    )
    
    fig.update_layout(
        title=title, scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'), margin=dict(t=40, l=0, b=0, r=0)
    )
    
    fig.show()

plot_image_with_mask_3D(image, true_mask, 'ground truth', channel=0)
plot_image_with_mask_3D(image, pred_mask, 'prediction', channel=0)